## Transformer Implementation

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

# downloading Shakespeare
import urllib.request
url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
text = urllib.request.urlopen(url).read().decode('utf-8')

print(f"Total characters: {len(text)}")
print(text[:200])

Total characters: 1115394
First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you


In [2]:
# vocabulary
chars = sorted(set(text))
vocab_size = len(chars)
char_to_idx = {c: i for i, c in enumerate(chars)}
idx_to_char = {i: c for c, i in char_to_idx.items()}

print(f"Vocab size: {vocab_size}")

# encoding text as integers
data = torch.tensor([char_to_idx[c] for c in text], dtype=torch.long)

# train/val split
n = int(0.9 * len(data))
train_data = data[:n]
val_data = data[n:]

print(f"Train tokens: {len(train_data)}")
print(f"Val tokens:   {len(val_data)}")

# hyperparameters
block_size = 64    # context length
batch_size = 32    # sequences per batch
d_model = 64       # embedding dimension
n_heads = 4        # attention heads
n_layers = 4       # transformer blocks
dropout = 0.1
lr = 3e-4
device = 'cuda' if torch.cuda.is_available() else 'cpu'

print(f"Device: {device}")

# batch generation
def get_batch(split):
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    return x.to(device), y.to(device)

# testing
xb, yb = get_batch('train')
print(f"Input batch shape:  {xb.shape}")
print(f"Target batch shape: {yb.shape}")

Vocab size: 65
Train tokens: 1003854
Val tokens:   111540
Device: cpu
Input batch shape:  torch.Size([32, 64])
Target batch shape: torch.Size([32, 64])


In [3]:
# token and positional embeddings
class GPT(nn.Module):
    def __init__(self):
        super().__init__()
        # token embedding
        self.token_embedding = nn.Embedding(vocab_size, d_model)
        # positional embedding
        self.position_embedding = nn.Embedding(block_size, d_model)
        # rest of the model
        self.lm_head = nn.Linear(d_model, vocab_size)

    def forward(self, x):
        B, T = x.shape
        
        tok_emb = self.token_embedding(x)
        pos = torch.arange(T, device=device)
        pos_emb = self.position_embedding(pos)

        x = tok_emb + pos_emb
        
        x = self.lm_head(x)
        
        return x

In [4]:
model = GPT().to(device)
xb, yb = get_batch('train')
out = model(xb)
print(f"Output shape: {out.shape}")

Output shape: torch.Size([32, 64, 65])


In [5]:
# single-head attention
class Head(nn.Module):
    def __init__(self, head_size):
        super().__init__()
        self.key   = nn.Linear(d_model, head_size, bias=False)
        self.query = nn.Linear(d_model, head_size, bias=False)
        self.value = nn.Linear(d_model, head_size, bias=False)
        self.dropout = nn.Dropout(dropout)
        
        self.register_buffer('tril', 
            torch.tril(torch.ones(block_size, block_size)))

    def forward(self, x):
        B, T, C = x.shape
        k = self.key(x)
        q = self.query(x)
        v = self.value(x)

        # attention scores
        scores = q @ k.transpose(-2, -1) / (C ** 0.5)  # (B, T, T)
        
        scores = scores.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
        
        weights = F.softmax(scores, dim=-1)
        weights = self.dropout(weights)
        
        output = weights @ v
        return output

In [6]:
# multi-head attention
class MultiHeadAttention(nn.Module):
    def __init__(self, n_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(n_heads)])
        self.proj = nn.Linear(d_model, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        # running all heads in parallel and concatenating the outputs
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        out = self.dropout(self.proj(out))
        return out

In [7]:
# feedforward network
class FeedForward(nn.Module):
    def __init__(self, d_model):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, 4 * d_model),
            nn.ReLU(),
            nn.Linear(4 * d_model, d_model),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)

In [8]:
# the transformer
class Block(nn.Module):
    def __init__(self, d_model, n_heads):
        super().__init__()
        head_size = d_model // n_heads
        self.attention = MultiHeadAttention(n_heads, head_size)
        self.ffn = FeedForward(d_model)
        self.ln1 = nn.LayerNorm(d_model)
        self.ln2 = nn.LayerNorm(d_model)

    def forward(self, x):
        # residual connections around each sublayer
        x = x + self.attention(self.ln1(x))
        x = x + self.ffn(self.ln2(x))
        return x

In [9]:
# gpt model
class GPT(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_embedding = nn.Embedding(vocab_size, d_model)
        self.position_embedding = nn.Embedding(block_size, d_model)
        self.blocks = nn.Sequential(*[Block(d_model, n_heads) for _ in range(n_layers)])
        self.ln_f = nn.LayerNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size)

    def forward(self, x, targets=None):
        B, T = x.shape
        tok_emb = self.token_embedding(x)
        pos_emb = self.position_embedding(torch.arange(T, device=device))
        x = tok_emb + pos_emb
        x = self.blocks(x)
        x = self.ln_f(x)
        logits = self.lm_head(x)

        loss = None
        if targets is not None:
            B, T, C = logits.shape
            loss = F.cross_entropy(logits.view(B*T, C), targets.view(B*T))

        return logits, loss

    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -block_size:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :]
            probs = F.softmax(logits, dim=-1)
            next_idx = torch.multinomial(probs, num_samples=1)
            idx = torch.cat([idx, next_idx], dim=1)
        return idx

In [ ]:
model = GPT().to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=lr)

# count parameters
n_params = sum(p.numel() for p in model.parameters())
print(f"Parameters: {n_params:,}")

@torch.no_grad()
def estimate_loss(eval_iters=100):
    model.eval()
    losses = {}
    for split in ['train', 'val']:
        split_losses = []
        for _ in range(eval_iters):
            x, y = get_batch(split)
            _, loss = model(x, y)
            split_losses.append(loss.item())
        losses[split] = np.mean(split_losses)
    model.train()
    return losses

# training
for step in range(3000):
    xb, yb = get_batch('train')
    logits, loss = model(xb, yb)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if step % 500 == 0:
        losses = estimate_loss()
        print(f"Step {step}: train={losses['train']:.4f}, val={losses['val']:.4f}")

# generating
model.eval()
context = torch.zeros((1, 1), dtype=torch.long, device=device)
generated = model.generate(context, max_new_tokens=300)
print(idx_to_char[0].join([idx_to_char[i.item()] for i in generated[0]]))

Parameters: 211,777
Step 0: train=4.2540, val=4.2538
Step 500: train=2.5008, val=2.5104
Step 1000: train=2.3586, val=2.3767
Step 1500: train=2.2521, val=2.2725
Step 2000: train=2.1657, val=2.1856
Step 2500: train=2.0936, val=2.1233


I
n
 
t
h
a
t
 
a
 
d
e
 
h
e
r
m
b
l
e
o
o
m
e
m
,
 
i
n
d
s
 
f
r
e
s


d
o
n
 
c
a
i
e
.




B
R
P
O
L
E
N
 
U
S
e
 
t
o
 
t
h
e
l
l
 
s
o
v
e
m
f
a
i
d
e
e
s
!


T
h
i
s
t
h
e
 
h
o
m
 
C
o
u
l
d
 
f
o
u
r
e
,
 
j
u
r
a
v
e
 
a
t
t
e
r
 
y
o
u
 
a
r
l
e
s
,


A
n
d
 
b
y
 
k
i
s
t
 
a
n
d
 
t
h
a
n
 
s
a
l
e
 
e
a
c
k
?




B
o
l
l
a
w
 
s
e
s
t
 
n
o
n
!


H
a
v
e
i
n
,
 
w
i
s
h
 
t
h
o
u
n
 
p
l
u
c
k
e
 
t
h
e
,
 
a
d
 
s
h
e
,
 
b
r
e
d
!


T
h
i
t
 
w
e
 
c
e
e
e
 
r
a
s
 
h
i
s
h
t
 
o
r
 
t
o
 
t
h
i
m
'
t
;


B
e
 
h
o
w
r
a
m
 
e
c
l
l
i
k
e
 
l
e
 
h
e
e
e
.




M
I
n
d
 
T
h
r


In [11]:
generated_text = ''.join([idx_to_char[i.item()] for i in generated[0]])
print(generated_text)


In that a de hermbleoomem, inds fres
don caie.

BRPOLEN USe to thell sovemfaidees!
Thisthe hom Could foure, jurave atter you arles,
And by kist and than sale eack?

Bollaw sest non!
Havein, wish thoun plucke the, ad she, bred!
Thit we ceee ras hisht or to thim't;
Be howram ecllike le heee.

MInd Thr


In [12]:
# training for longer
for step in range(3000, 8000):
    xb, yb = get_batch('train')
    logits, loss = model(xb, yb)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if step % 1000 == 0:
        losses = estimate_loss()
        print(f"Step {step}: train={losses['train']:.4f}, val={losses['val']:.4f}")

generated = model.generate(torch.zeros((1,1), dtype=torch.long, device=device), 500)
print(''.join([idx_to_char[i.item()] for i in generated[0]]))

Step 3000: train=2.0308, val=2.0793
Step 4000: train=1.9241, val=1.9975
Step 5000: train=1.8555, val=1.9449
Step 6000: train=1.7954, val=1.9084
Step 7000: train=1.7528, val=1.8793

First upon dea;
How the my ture frow wh an; whickesome! I
And dut Ind thour here truow this!' is so,
And if a my shall such pomotes.

LORS ABEO:
And my now, carre; thee than had woult an share wher abrochichy
dom itfor, my pars yin!

For
MEgINCA:
How in with is of broble thenk tan the the
shiBe In so make mose when thou a with.

Prought:
I bre'd, las which adlow, hare you him rok statil
Pors tauble tink where ageingood turg?
Sifbere; breseser con, of charid fure, graws as and raince?

RINVABE:
W
